#A Positional Encoding Example
Copyright 2023 Denis Rothman, MIT License

Reference 1 for Positional Encoding:
Attention is All You Need paper, page 6,Google Brain and Google Research

Reference 2 for word embedding:
https://www.geeksforgeeks.org/python-word-embedding-using-word2vec/
Reference 3 for cosine similarity:
SciKit Learn cosine similarity documentation

The goal of this notebook is to understand positional encoding and cosine similarity. Cosine similarity remains a solid NLP approach.

The text.txt file is just to illustrate the concepts in the notebook.

Downloading text file

In [1]:
#text.file
!curl -L https://raw.githubusercontent.com/Denis2054/Transformers-for-NLP-and-Computer-Vision-3rd-Edition/master/Chapter02/text.txt --output "text.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 24151  100 24151    0     0  55107      0 --:--:-- --:--:-- --:--:-- 55265


In [2]:
#!pip install gensim # Version Gensim 4.0.0 and above
import torch
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\westw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [82]:
import math
import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
import gensim
from gensim.models import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings(action = 'ignore')


dprint=1 # prints outputs if set to 1, default=0

# read ‘text.txt’ file
sample = open("text.txt", "r")
s = sample.read()

# processing escape characters
f = s.replace("\n", " ")

data = []

# sentence parsing
for i in sent_tokenize(f):
	temp = []
	# tokenize the sentence into words
	for j in word_tokenize(i):
		temp.append(j.lower())
	data.append(temp)

# Creating Skip Gram model
vector_size=6


model2 = gensim.models.Word2Vec(data, min_count = 1, vector_size = vector_size, window = 5, sg = 1)

In [ ]:
# below shows the relationship between the word-pair in two phrases
# "black cat" and "brown dog"
# all possible combinations of words in the phrases
# "black - brown", "black - dog", "cat - brown", "cat - dog" (which is C(2,2)=4)

def readFromInput():
    word1=input("Enter first word: ")
    word2=input("Enter second word: ")
    pos1=int(input("Enter position of first word: "))
    pos2=int(input("Enter position of second word: "))
    return word1,word2,pos1,pos2

word1 = "black"
word2 = "brown"
phrase1 = "black cat"
phrase2 = "brown dog"
pos1 = 2
pos2 = 10
#word1,word2,pos1,pos2=readFromInput()


a = model2.wv[word1]
b = model2.wv[word2]

phrase_a = model2.wv[phrase1.split()]
phrase_b = model2.wv[phrase2.split()]


phrase1_arr = phrase1.split()
phrase2_arr = phrase2.split()
print(f'word1: {word1}, word2: {word2}')
print(f'word1 vector: {a}')
print(f'word2 vector: {b}')
print(f'phrase_a: {phrase_a}')
print(f'phrase_b: {phrase_b}')
print(f'position of word1: {pos1}, position of word2: {pos2}')

if(dprint==1):
        print(a)

# compute cosine similarity
dot = np.dot(a, b)
norma = np.linalg.norm(a)
normb = np.linalg.norm(b)
cos = dot / (norma * normb)

aa = a.reshape(1,vector_size)
ba = b.reshape(1,vector_size)
cos_lib = cosine_similarity(aa, ba)

# mean(axis=0) averages features across words to get a single phrase vector. [[3,1][2,5]] -> [2.5, 3]
phrase_aa = phrase_a.mean(axis=0).reshape(1,vector_size) 
phrase_ba = phrase_b.mean(axis=0).reshape(1,vector_size)
phrase_cos_lib = cosine_similarity(phrase_aa, phrase_ba)

print(f'Cosine similarity between phrases "{phrase1}" and "{phrase2}" is {phrase_cos_lib[0][0]}')

if dprint==1:
        print(f'Cosine similarity between {word1} and {word2} is {cos}')
        print(f'Cosine similarity using sklearn library between {word1} and {word2} is {cos_lib}')
        
'''
Why the cosine similarity between "black" and "cat"(0.98789) is higher than that between "black" and "brown" (=0.97475)?
Because "black" and "cat" appear together in the same context (the phrase "the black cat") more frequently than "black" and "brown" do. In the provided text, "black" is directly associated with "cat," while "brown" is associated with "dog." This co-occurrence in similar contexts leads to a higher cosine similarity score between "black" and "cat." Also the models will learn about other features such as color, size, shape etc. which will also contribute to the similarity score.
'''

phrase1_matrix = np.zeros((len(phrase1_arr), model2.vector_size)) # create a zero matrix for phrase1
phrase2_matrix = np.zeros((len(phrase1_arr), model2.vector_size)) # create a zero matrix for phrase2

for i in range(len(phrase1_arr)):
    wv = model2.wv[phrase1_arr[i]]
    norm = np.linalg.norm(wv) # calc the norm 
    # insert wv to replace the word vector in aa
    phrase1_matrix[i] = wv
    if dprint==1:
        print(f'phrase1Arr[{i}] norm: {norm}')
        
for i in range(len(phrase2_arr)):
    wv = model2.wv[phrase2_arr[i]]
    norm = np.linalg.norm(wv) # calc the norm 
    # insert wv to replace the word vector in bb
    phrase2_matrix[i] = wv
    # if dprint==1:
    #     print(f'phrase2Arr[{i}] norm: {norm}')
    
print(f'phrase1_matrix: {phrase1_matrix}')
print(f'phrase2_matrix: {phrase2_matrix}')


word1: black, word2: brown
word1 vector: [-0.44209355 -0.22209145  0.64672685 -0.08660428 -0.21000132  0.85100114]
word2 vector: [-0.15330462 -0.22514266  0.7495122  -0.2075421  -0.16728257  0.86467797]
phrase_a: [[-0.44209355 -0.22209145  0.64672685 -0.08660428 -0.21000132  0.85100114]
 [-0.47344136 -0.0664994   0.6828494  -0.14041716 -0.1103754   0.87191   ]]
phrase_b: [[-0.15330462 -0.22514266  0.7495122  -0.2075421  -0.16728257  0.86467797]
 [-0.46203992 -0.20224676  0.88407445 -0.17850259 -0.14484864  0.6681052 ]]
position of word1: 2, position of word2: 10
[-0.44209355 -0.22209145  0.64672685 -0.08660428 -0.21000132  0.85100114]
Cosine similarity between phrases "black cat" and "brown dog" is 0.9773752689361572
Cosine similarity between black and brown is 0.9617871642112732
Cosine similarity using sklearn library between black and brown is [[0.9617872]]
phrase1Arr[0] norm: 1.1995127201080322
phrase1Arr[1] norm: 1.2194174528121948
phrase1_matrix: [[-0.44209355 -0.22209145  0.64672

In [78]:
print(f'word1: {word1}, word2: {word2}')
print(f'word1 vector: {a}')
print(f'word2 vector: {b}')

print(f'reshaped  [1,512] version of a - aa: {aa}')
print(f'reshaped  [1,512] version of b - ba: {ba}')

word1: black, word2: brown
word1 vector: [-0.44209355 -0.22209145  0.64672685 -0.08660428 -0.21000132  0.85100114]
word2 vector: [-0.15330462 -0.22514266  0.7495122  -0.2075421  -0.16728257  0.86467797]
reshaped  [1,512] version of a - aa: [[-0.44209355 -0.22209145  0.64672685 -0.08660428 -0.21000132  0.85100114]]
reshaped  [1,512] version of b - ba: [[-0.15330462 -0.22514266  0.7495122  -0.2075421  -0.16728257  0.86467797]]


A Positional Encoding example using one line of basic Python using a few lines of code for the sine and cosine functions.
I added a Pytorch method inspired by Pytorch.org to explore these methods.
The main idea to keep in mind is that we are looking to add small values to the word embedding output so that the positions are taken into account. This means that as long as the cosine similarity, for example, displayed at the end of the notebook, shows the positions have been taken into account, the method can apply. Depending on the Transformer model, this method can be fine-tuned as well as using other methods.

In [84]:
pe1=aa.copy()
pe2=aa.copy()
pe3=aa.copy()
paa=aa.copy()
pba=ba.copy()
d_model=vector_size
max_print=d_model
#max_length=20

print(f'max_print: {max_print}')
print(f'aa: {aa}')

for i in range(0, max_print,2):
                pe1[0][i] = math.sin(pos1 / (10000 ** ((2 * i)/d_model)))
                paa[0][i] = (paa[0][i]*math.sqrt(d_model))+ pe1[0][i]
                pe1[0][i+1] = math.cos(pos1 / (10000 ** ((2 * i)/d_model)))
                paa[0][i+1] = (paa[0][i+1]*math.sqrt(d_model))+pe1[0][i+1]
                if dprint==1:
                        print(i,pe1[0][i],i+1,pe1[0][i+1])
                        print(i,paa[0][i],i+1,paa[0][i+1])
                        print("\n")

# #print(pe1)
# # Example: A method in Pytorch using torch.exp and math.log :
# max_len=max_length
# pe = torch.zeros(max_len, d_model)
# position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
# div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
# pe[:, 0::2] = torch.sin(position * div_term) # [0::2] select even indices, : for all rows, x::y for every y-th element starting from x
# pe[:, 1::2] = torch.cos(position * div_term) # [1::2] select odd indices
# print(pe[:, 0::2])

max_print: 6
aa: [[-0.44209355 -0.22209145  0.64672685 -0.08660428 -0.21000132  0.85100114]]
0 0.9092974 1 -0.41614684
0 -0.17360622 1 -0.9601576


2 0.004308856 3 0.9999907
2 1.5884596 3 0.7878544


4 9.283178e-06 5 1.0
4 -0.5143868 5 3.0845187




In [85]:

for i in range(0, max_print,2):
                pe2[0][i] = math.sin(pos2 / (10000 ** ((2 * i)/d_model)))
                pba[0][i] = (pba[0][i]*math.sqrt(d_model))+ pe2[0][i]

                pe2[0][i+1] = math.cos(pos2 / (10000 ** ((2 * i)/d_model)))
                pba[0][i+1] = (pba[0][i+1]*math.sqrt(d_model))+ pe2[0][i+1]

                # if dprint==1:
                #         print(i,pe2[0][i],i+1,pe2[0][i+1])
                #         print(i,paa[0][i],i+1,paa[0][i+1])
                        
                #         print("\n")

print(word1,word2)

print(f'pe1: {pe1}')
print(f'pe2: {pe2}')
print(f'paa: {paa}')
print(f'pba: {pba}')
cos_lib = cosine_similarity(aa, ba)
print(cos_lib,"word similarity")
cos_lib = cosine_similarity(pe1, pe2)
print(cos_lib,"positional similarity")
cos_lib = cosine_similarity(paa, pba)
print(cos_lib,"positional encoding similarity")




black brown
pe1: [[ 9.0929741e-01 -4.1614684e-01  4.3088561e-03  9.9999070e-01
   9.2831779e-06  1.0000000e+00]]
pe2: [[-5.4402113e-01 -8.3907151e-01  2.1542680e-02  9.9976796e-01
   4.6415887e-05  1.0000000e+00]]
paa: [[-0.17360622 -0.9601576   1.5884596   0.7878544  -0.5143868   3.0845187 ]]
pba: [[-0.9195392  -1.3905561   1.8574651   0.4913957  -0.40971053  3.1180198 ]]
[[0.9617872]] word similarity
[[0.61811715]] positional similarity
[[0.9730655]] positional encoding similarity


In [ ]:
# positional coding for phrase1 and phrase2
pos1_arr = [2,3]
pos2_arr = [10,11]


pe1_phrase=phrase1_matrix.copy()
pe2_phrase=phrase1_matrix.copy()
pe3_phrase=phrase1_matrix.copy()
paa_phrase=phrase1_matrix.copy()
pba_phrase=phrase1_matrix.copy()


d_model=vector_size
max_print=d_model


for row_index in range(0, len(pos1_arr)):
    for i in range(0, max_print, 2):
        pe1_phrase[row_index][i] = math.sin(pos1_arr[row_index] / (10000 ** ((2 * i)/d_model)))
        paa_phrase[row_index][i] = (paa[0][i]*math.sqrt(d_model))+ pe1[0][i]
        pe1_phrase[row_index][i+1] = math.cos(pos1_arr[row_index]  / (10000 ** ((2 * i)/d_model)))
        paa_phrase[row_index][i+1] = (paa[0][i+1]*math.sqrt(d_model))+pe1[0][i+1]
        # if dprint==1:
        #     print(i,pe1[row_index][i],i+1,pe1[0][i+1])
        #     print(i,paa[row_index][i],i+1,paa[0][i+1])
        #     print("\n")

# print(f'phrase1: {phrase1}, phrase2: {phrase2}')
# print(f'phrase1 vector: {paa}')
# print(f'phrase2 vector: {pba}')

for row_index in range(0, len(pos2_arr)):
    for i in range(0, max_print, 2):
        pe2_phrase[row_index][i] = math.sin(pos2_arr[row_index]  / (10000 ** ((2 * i)/d_model)))
        pba_phrase[row_index][i] = (pba[0][i]*math.sqrt(d_model))+ pe2[0][i]

        pe2_phrase[row_index][i+1] = math.cos(pos2_arr[row_index]  / (10000 ** ((2 * i)/d_model)))
        pba_phrase[row_index][i+1] = (pba[0][i+1]*math.sqrt(d_model))+ pe2[0][i+1]

        # if dprint==1:
        #     print(i,pe2[row_index][i],i+1,pe2[0][i+1])
        #     print(i,paa[row_index][i],i+1,paa[0][i+1])
        #     print("\n")

print(f'pe1_phrase: {pe1}')
print(f'pe2_phrase: {pe2}')
print(f'paa_phrase: {paa}')
print(f'pba_phrase: {pba}')

cos_lib = cosine_similarity(phrase1_matrix, phrase2_matrix)
print(cos_lib,"phrase similarity")
cos_lib = cosine_similarity(pe1_phrase, pe2_phrase)
print(cos_lib,"positional similarity")
cos_lib = cosine_similarity(paa_phrase, pba_phrase)
print(cos_lib,"positional encoding similarity")


pe1_phrase: [[ 9.0929741e-01 -4.1614684e-01  4.3088561e-03  9.9999070e-01
   9.2831779e-06  1.0000000e+00]]
pe2_phrase: [[-5.4402113e-01 -8.3907151e-01  2.1542680e-02  9.9976796e-01
   4.6415887e-05  1.0000000e+00]]
paa_phrase: [[-0.17360622 -0.9601576   1.5884596   0.7878544  -0.5143868   3.0845187 ]]
pba_phrase: [[-0.9195392  -1.3905561   1.8574651   0.4913957  -0.40971053  3.1180198 ]]
[[0.96178721 0.96578396]
 [0.95250077 0.96596712]] phrase similarity
[[0.61811715 0.36289392]
 [0.91792951 0.61811715]] positional similarity
[[0.94381372 0.94381372]
 [0.94381372 0.94381372]] positional encoding similarity
